In [1]:
import numpy as np
import pandas as pd
import subprocess
import random
from string import Template
from tempfile import NamedTemporaryFile
from multiprocessing import Pool
import glob
from tqdm import tqdm
from scipy import optimize
PATH = "/home/jmurga/mkt/202004/rawData"

In [ ]:
from numba import njit
import numpy as np

def amkt(daf, div, xlow, xhigh,check='raise'):
    res = {}

    d_ratio = float(div[1] / div[0])
    # Compute alpha values and trim
    alpha = 1 - d_ratio * (daf[:,1] / daf[:,2])
    trim = ((daf[:,0] >= xlow) & (daf[:,0] <= xhigh))

    # Two-step model fit:
    # First bounded fit:
    try:
        popt, pcov = optimize.curve_fit(exp_model, daf[:,0][trim], alpha[trim],bounds=([-1, -1, 1], [1, 1, 10]))
        # print('fit initial')
    except:
        # print('could not fit initial')
        popt = None
        pcov = None


    # Second fit using initially guessed values or unbounded fit:
    try:
        popt, pcov = optimize.curve_fit(exp_model, daf[:,0][trim], alpha[trim], p0=popt,method='lm')
        # print('Fit: lm')
    except:
        try:
            popt, pcov = optimize.curve_fit(exp_model, daf[:,0][trim], alpha[trim], p0=popt,  method='trf')
            # print('Fit: trf')
        except:
            try:
                popt, pcov = optimize.curve_fit(exp_model, daf[:,0][trim], alpha[trim], p0=popt, method='dogbox')
                # print('Fit: dogbox')
            except:
                if not popt:
                    # print('Could not fit any unbounded')
                    raise RuntimeError("Couldn't fit any method")

    res['a'] = popt[0]
    res['b'] = popt[1]
    res['c'] = popt[2]

    # alpha for predicted model
    res['alpha'] = exp_model(1.0, res['a'], res['b'], res['c'])
    
    # Compute confidence intervals based on simulated data (MC-SOERP)
    vcov = pd.concat([pd.DataFrame([0] * 4).transpose(),
                      pd.concat([pd.DataFrame([0] * 4), pd.DataFrame(pcov)], axis=1, ignore_index=True)],
                     axis=0, ignore_index=True)
    vcov = vcov.iloc[0:4, :].values

    simpars = np.random.multivariate_normal(mean=[1.0, res['a'], res['b'], res['c']], cov=vcov, size=10000,
                                            check_valid=check)  # check_valid=raise -> same as R implementation

    res['ciLow'], res['ciHigh'] = np.quantile([exp_model(x[0], x[1], x[2], x[3]) for x in simpars], [0.025, 0.975])

    return res

@njit
def exp_model(f_trimmed, a, b, c):
    return a + b * np.exp(-c * f_trimmed)

@njit
def cumulativeSfs(x):

	f       = x[:,0]
	sfsTemp = x[:,1:]
	
	out       = np.empty_like(x)
	out[0,0]  = f[0]
	out[0,1:] = np.sum(sfsTemp,axis=0)

	for i in range(1,out.shape[0]):

		app = out[i-1,1:] - sfsTemp[i-1,:]

		if np.sum(app) > 0.0:
			out[i,0]  = f[i]
			out[i,1:] = app
		else:
			out[i,0]  = f[i]
			out[i,:] = np.zeros(app.shape[0])

	return out

def reduceSfs(x,bins):

    f = x[:,0]
    sfs = x[:,1:]
    
    b = np.arange(0,1,1/bins)
    inds = np.digitize(f,b,right=True)
    out  = np.zeros((bins,x.shape[1]))
    out[:,0]  = np.unique(inds)
    
    sfsGrouped = np.hstack([np.reshape(inds,(inds.shape[0],1)),sfs])
    for i in np.unique(inds):
        out[out[:,0]==i,1:] = np.sum(sfsGrouped[sfsGrouped[:,0] == i,1:],axis=0) 
        
    out[:,0] = b
    return(out)

In [1]:
def runSlim(recipe,wStrength,sStrength,N,pposL,pposH,output,iteration):
    """
    Runs SLiM using subprocess.
    Args:
        command_line_args: list; a list of command line arguments.
    return: The entire SLiM output as a string.
    """
    
    slimRecipe = Template(open(recipe, "r").read())

    mapping = {
        'weaklyStrength' : wStrength,
        'strongStrength' : sStrength,
        'N' : N,
        'pposL' : pposL,
        'pposH' : pposH
    }

    #print(slimRecipe.substitute(mapping))

    with NamedTemporaryFile("w") as slim_file:
        print(slimRecipe.substitute(mapping), file= slim_file,flush=True)

        slim = subprocess.run(["/home/jmurga/.conda/envs/abc-mk/bin/slim", "-s", str(random.randint(1, 10**13)),slim_file.name],universal_newlines=True,stdout=subprocess.PIPE)

        slimResults = slim.stdout.split('\n')
        slimDaf = slimResults[slimResults.index('daf\tpi\tp0\tpw'):slimResults.index('di\td0')]
        slimDiv = slimResults[slimResults.index('di\td0'):slimResults.index('trueAlphaW\ttrueAlphaS\ttrueAlpha')]
        slimAlpha = slimResults[slimResults.index('trueAlphaW\ttrueAlphaS\ttrueAlpha'):-1]

        # Extract daf info from slim results
        slimDaf = [x.split('\t') for x in slimDaf]

        # Header daf
        h = slimDaf[0]
        # Data
        d = slimDaf[1:]
        # Pandas dataframe
        daf = pd.DataFrame(d,columns=h)
        daf.daf = np.around(daf.daf.astype(float),4)

        # Extract divergence info from slim results
        slimDiv = [x.split('\t') for x in slimDiv]
        slimAlpha = [x.split('\t') for x in slimAlpha]

        # Header div
        h = slimDiv[0]
        # Data
        d = slimDiv[1]
        # Pandas dataframe
        div = pd.DataFrame(np.reshape(np.array(d),(1,2)),columns=h)

        # Header alpha
        h = slimAlpha[0]
        # data
        d = slimAlpha[1]
        alpha = pd.DataFrame(np.reshape(np.array(d),(1,3)),columns=h)

        # Pandas dataframe
        daf.to_csv(output + "/daf/daf" + str(iteration) + ".tsv.gz",index=False,sep="\t",compression="gzip")
        div.to_csv(output + "/div/div" + str(iteration) + ".tsv.gz",index=False,sep="\t",compression="gzip")
        alpha.to_csv(output + "/div/alpha" + str(iteration) + ".tsv.gz",index=False,sep="\t",compression="gzip")
        
    # Parsing string output, we checked position on slim custom printed output and procces each variable taking into account correspondent positions. Excluding recipe execution info

In [18]:
def inputSlim(iterations,recipe,wStrength,sStrength,N,pposL,pposH,output):
    
    totalSim = iterations[1]-iterations[0] +1
    irecipe = [recipe] * totalSim
    iwStrength = [wStrength] * totalSim
    isStrength = [sStrength] * totalSim
    iN = [N] * totalSim
    ipposL = [pposL] * totalSim
    ipposH = [pposH] * totalSim
    ioutput = [output] * totalSim

    total = [i for i in range(iterations[0],iterations[1] + 1)]
    
    inp = list(zip(irecipe,iwStrength,isStrength,iN,ipposL,ipposH,ioutput,total))
    
    return(inp)

In [4]:
def parsePolDiv(dafPath,divPath):
    """
    slr, tupple array with daf and div data by element in list
    """
    
    dafFiles = glob.glob(dafPath + "/*.tsv")
    divFiles = glob.glob(divPath + "/div*.tsv")
    alFiles  = glob.glob(divPath + "/al*.tsv")

    iteration = len(dafFiles)
    sfs       = pd.DataFrame(np.zeros((1321,4)),columns=['pi','p0','pw','pi_nopos'])
    divs      = pd.DataFrame(np.zeros((1,2)),columns=['di','d0'])
    alphas    = pd.DataFrame(np.zeros((iteration,3)),columns=['trueAlphaW','trueAlphaS','trueAlpha'])

    for f in tqdm(range(0,iteration)):
        daf             = pd.read_csv(dafFiles[f], header=0, sep='\t')
        daf['pi_nopos'] = daf.pi - daf.pw
        sfs           = sfs + daf.iloc[:,1:]
        
        d = pd.read_csv(divFiles[f], header=0, sep='\t')  
        divs = divs + d
        
        al = pd.read_csv(alFiles[f], header=0, sep='\t')  
        alphas.iloc[f] = al.to_numpy()
        
    sfs.insert(0,'f',daf.iloc[:,0])
    # div = np.sum(divs,axis=0)
        
    return(sfs,divs,alphas)

In [ ]:
bgs = inputSlim((0,10000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgs.slim",10,500,1000,1,"/home/jmurga/mkt/202004/rawData/simulations/noBgsNoDemog")

In [ ]:
p = Pool(processes=10)
output = p.starmap(runSlim,bgs)
p.terminate()

In [ ]:
sfs, div, alphas = parsePolDiv(PATH + "/simulations/noBgsNoDemog/daf/", PATH + "/simulations/noBgsNoDemog/div")

Proportion

In [ ]:
bgs = inputSlim((0,10000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgsVar.slim",10,500,1000,0.00387663,0.000311471,"/home/jmurga/mkt/202004/rawData/simulations/noBgsNoDemogProp")

In [ ]:
p = Pool(processes=10)
output = p.starmap(runSlim,bgs)
p.terminate()

In [ ]:
sfs, div, alphas = parsePolDiv(PATH + "/simulations/noBgsNoDemogProp/daf/", PATH + "/simulations/noBgsNoDemogProp/div")

In [ ]:
sfs.iloc[:,[1,2,3,4]].to_csv(PATH + "/simulations/noBgsNoDemog/sfsBgs.tsv",header=True,index=False,sep="\t")
div.to_csv(PATH + "/simulations/noBgsNoDemog/divBgs.tsv",header=True,index=False,sep="\t")
alphas.to_csv(PATH + "/simulations/noBgsNoDemog/alphasBgs.tsv",header=True,index=False,sep="\t")

In [ ]:
sSfs.iloc[:,[1,2,3,4]].to_csv(PATH + "/simulations/noBgsNoDemog/sfsBgs.tsv",header=True,index=False,sep="\t")

In [ ]:
cSfs = cumulativeSfs(sfs.to_numpy())
crSfs = cumulativeSfs(rSfs.to_numpy())

In [ ]:
amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)